In [1]:
!pip install openai requests jsonschema

In [2]:
import os
import re
import json
import joblib
import requests
import pandas as pd

from jsonschema import validate, ValidationError

In [4]:
model = joblib.load("best_model.pkl")

print(model)

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=10, n_estimators=50,
                                        random_state=42))])


In [5]:
import os

os.environ["LLM_API_KEY"] = "YOUR_API_KEY"

api_key = os.environ["LLM_API_KEY"]

In [6]:
url = "YOUR_API_ENDPOINT"

def call_llm(system_prompt,
             user_prompt,
             temperature=0,
             max_tokens=512):

    payload = {
        "model":"YOUR_MODEL",
        "messages":[
            {
                "role":"system",
                "content":system_prompt
            },
            {
                "role":"user",
                "content":user_prompt
            }
        ],
        "temperature":temperature,
        "max_tokens":max_tokens
    }

    headers = {
        "Authorization":f"Bearer {api_key}",
        "Content-Type":"application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:

        print(response.status_code)

        return None

    return response.json()["choices"][0]["message"]["content"]

In [10]:
import os
from getpass import getpass

os.environ["LLM_API_KEY"] = getpass("Enter your OpenRouter API Key: ")

api_key = os.environ["LLM_API_KEY"]

url = "https://openrouter.ai/api/v1/chat/completions"

Enter your OpenRouter API Key: ··········


In [16]:
import requests

url = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    payload = {
        "model": "openai/gpt-4.1-mini",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("Status Code:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [17]:
system_prompt = "Reply only with the word: hello"

user_prompt = "Say hello."

print(call_llm(system_prompt, user_prompt))

hello


In [18]:
from jsonschema import validate, ValidationError

schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

In [19]:
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

In [20]:
print(has_pii("Contact me at abc@gmail.com"))
print(has_pii("Toyota Camry 2022"))

True
False


In [21]:
import joblib

model = joblib.load("best_model.pkl")

print("Model loaded successfully!")

Model loaded successfully!


In [23]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")

In [25]:
df["price_category"] = (df["price"] > df["price"].median()).astype(int)

In [26]:
X = df.drop(columns=["price", "price_category"])
y = df["price_category"]

In [28]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")   # Replace with your CSV filename if different

y_reg = df["price"]
y_clf = (y_reg > y_reg.median()).astype(int)

X = df.drop(columns=["price"])

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg,
    test_size=0.2,
    random_state=42
)

_, _, y_clf_train, y_clf_test = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42
)

In [30]:
sample1 = X_test.iloc[[0]]
sample2 = X_test.iloc[[1]]
sample3 = X_test.iloc[[2]]

samples = [sample1, sample2, sample3]

In [31]:
import joblib

model = joblib.load("best_model.pkl")

print(model)

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=10, n_estimators=50,
                                        random_state=42))])


In [33]:
print(X.columns.tolist())

['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']


In [34]:
print(model.feature_names_in_)

['model_year' 'milage' 'brand_Alfa' ... 'int_col_Yellow' 'int_col_–'
 'accident_None reported']


In [35]:
X_encoded = pd.get_dummies(X)

In [36]:
X_encoded = X_encoded.reindex(
    columns=model.feature_names_in_,
    fill_value=0
)

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X_encoded,
    y_clf,
    test_size=0.2,
    random_state=42
)

In [38]:
sample1 = X_test.iloc[[0]]
sample2 = X_test.iloc[[1]]
sample3 = X_test.iloc[[2]]

samples = [sample1, sample2, sample3]

In [39]:
for i, sample in enumerate(samples, start=1):

    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0].max()

    print(f"Sample {i}")
    print("Prediction:", prediction)
    print("Probability:", probability)

Sample 1
Prediction: 0
Probability: 0.5253800583044206
Sample 2
Prediction: 0
Probability: 0.616640280492199
Sample 3
Prediction: 0
Probability: 0.5520212631389808


In [40]:
system_prompt = """
You are an AI assistant that explains machine learning predictions.

Return ONLY valid JSON with the following fields:

{
  "prediction_label": "",
  "confidence_level": "",
  "top_reason": "",
  "second_reason": "",
  "next_step": ""
}
"""

In [41]:
import json

for i, sample in enumerate(samples, start=1):

    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0].max()

    user_prompt = f"""
Feature Values:
{sample.to_dict()}

Predicted Class: {prediction}

Predicted Probability: {probability:.4f}
"""

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        continue

    response = call_llm(system_prompt, user_prompt, temperature=0)

    print(f"\nSample {i}")
    print("Raw Response:")
    print(response)

    try:
        explanation = json.loads(response.strip())
        validate(instance=explanation, schema=schema)
        print("Validation: PASS")
    except json.JSONDecodeError:
        print("Validation: FAIL - Invalid JSON")
    except ValidationError as e:
        print("Validation: FAIL")
        print(e)


Sample 1
Raw Response:
{
  "prediction_label": "Class 0",
  "confidence_level": "52.54%",
  "top_reason": "The vehicle is a 2018 model year Lexus IS 300 with gasoline fuel type, which aligns with characteristics of Class 0.",
  "second_reason": "The mileage is moderate at 50,992, and the transmission is automatic, both factors contributing to the prediction of Class 0.",
  "next_step": "Consider reviewing additional vehicle details or obtaining more data to increase prediction confidence."
}
Validation: PASS

Sample 2
Raw Response:
{
  "prediction_label": "0",
  "confidence_level": "61.66%",
  "top_reason": "The vehicle is a 2004 Chevrolet with 64,500 miles, which aligns with typical characteristics of class 0 in the model.",
  "second_reason": "The fuel type is gasoline and the transmission is automatic, both common features contributing to the prediction of class 0.",
  "next_step": "Consider verifying the vehicle's condition and history to confirm the prediction and assess if furth

In [42]:
sample = samples[0]

prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0].max()

user_prompt = f"""
Feature Values:
{sample.to_dict()}

Predicted Class: {prediction}

Predicted Probability: {probability:.4f}
"""

print("Temperature = 0")
print(call_llm(system_prompt, user_prompt, temperature=0))

print("\nTemperature = 0.7")
print(call_llm(system_prompt, user_prompt, temperature=0.7))

Temperature = 0
{
  "prediction_label": "0",
  "confidence_level": "52.54%",
  "top_reason": "The car is a 2018 Lexus IS 300 with gasoline fuel type, which is a common and reliable model.",
  "second_reason": "The mileage is moderate at 50,992, and the transmission is automatic, both factors that typically contribute to a stable prediction.",
  "next_step": "Consider verifying additional vehicle details or obtaining more data to increase prediction confidence."
}

Temperature = 0.7
{
  "prediction_label": "Class 0",
  "confidence_level": "52.54%",
  "top_reason": "The vehicle's brand is Lexus, which influences the prediction towards Class 0.",
  "second_reason": "The engine specification of 260.0HP 3.5L V6 Cylinder Engine Gasoline Fuel supports the likelihood of Class 0.",
  "next_step": "Review additional vehicle features or collect more data to improve prediction confidence."
}


In [43]:
blocked_input = "Contact me at student@example.com"

if has_pii(blocked_input):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, blocked_input))

Input blocked: PII detected.


In [44]:
clean_input = "Used car with 45000 miles and model year 2021."

if has_pii(clean_input):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, clean_input))

{
  "prediction_label": "Good Condition",
  "confidence_level": "High",
  "top_reason": "Low mileage for a 2021 model year, indicating less wear and tear.",
  "second_reason": "Relatively recent model year suggests modern features and updated technology.",
  "next_step": "Conduct a detailed inspection and review the vehicle history report before purchase."
}


In [45]:
clean_input = "Used car with 45000 miles and model year 2021."

if has_pii(clean_input):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, clean_input))

{
  "prediction_label": "Good Condition",
  "confidence_level": "High",
  "top_reason": "Low mileage for a 2021 model year, indicating less wear and tear.",
  "second_reason": "Relatively recent model year suggests modern features and updated technology.",
  "next_step": "Check the vehicle history report and schedule a mechanical inspection."
}


In [46]:
sample = samples[0]

prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0].max()

user_prompt = f"""
Feature Values:
{sample.to_dict()}

Predicted Class: {prediction}

Predicted Probability: {probability:.4f}

Explain this prediction and return ONLY valid JSON.
"""

print("===== Temperature = 0 =====")
print(call_llm(system_prompt, user_prompt, temperature=0))

print("\n===== Temperature = 0.7 =====")
print(call_llm(system_prompt, user_prompt, temperature=0.7))

===== Temperature = 0 =====
{
  "prediction_label": "0",
  "confidence_level": "52.54%",
  "top_reason": "The car is a 2018 model Lexus IS 300 with gasoline fuel type, which typically has moderate resale value.",
  "second_reason": "The mileage is 50,992, which is average and does not strongly indicate either high or low value, leading to a moderate confidence in the prediction.",
  "next_step": "Consider verifying additional features such as accident history or interior condition to improve prediction confidence."
}

===== Temperature = 0.7 =====
{
  "prediction_label": "0",
  "confidence_level": "52.54%",
  "top_reason": "The car is a 2018 model Lexus IS 300 with gasoline fuel type, which typically aligns with class 0 characteristics in the model.",
  "second_reason": "The mileage is moderate at 50,992, which supports the prediction for class 0 as it fits the expected usage profile.",
  "next_step": "Consider reviewing additional features or obtaining more data to increase prediction

In [47]:
blocked_input = "My email is student@gmail.com"

if has_pii(blocked_input):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, blocked_input))

Input blocked: PII detected.


In [48]:
clean_input = "Used car with 45000 miles and model year 2021."

if has_pii(clean_input):
    print("Input blocked: PII detected.")
else:
    print(call_llm(system_prompt, clean_input))

{
  "prediction_label": "Good Condition",
  "confidence_level": "High",
  "top_reason": "Low mileage for a 2021 model year, indicating less wear and tear.",
  "second_reason": "Relatively recent model year suggests modern features and better reliability.",
  "next_step": "Conduct a detailed inspection and review the vehicle history report before purchase."
}


In [49]:
for i, sample in enumerate(samples, start=1):

    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0].max()

    print(f"\n========== Sample {i} ==========")
    print("Feature Values:")
    print(sample)

    print("\nPredicted Class:", prediction)
    print("Probability:", round(probability, 4))

    response = call_llm(system_prompt, str(sample.to_dict()), temperature=0)

    print("\nLLM Response:")
    print(response)


========== Sample 1 ==========
Feature Values:
      model_year  milage  brand_Alfa  brand_Aston  brand_Audi  brand_BMW  \
2580        2018   50992       False        False       False      False   

      brand_Bentley  brand_Bugatti  brand_Buick  brand_Cadillac  ...  \
2580          False          False        False           False  ...   

      int_col_Tupelo  int_col_Very Light Cashmere  int_col_WHITE  \
2580           False                        False          False   

      int_col_Walnut  int_col_Whisper Beige  int_col_White  \
2580           False                  False          False   

      int_col_White / Brown  int_col_Yellow  int_col_–  accident_None reported  
2580                  False           False      False                   False  

[1 rows x 3641 columns]

Predicted Class: 0
Probability: 0.5254

LLM Response:
{
  "prediction_label": "Likely to be sold",
  "confidence_level": "High",
  "top_reason": "The car is a 2018 Lexus IS 300 with gasoline fuel type, wh